In [25]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [26]:
daf=pd.read_csv("../data/raw/dataset_transacciones/hey_productos.csv")

In [27]:
daf.describe()

,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad
count,14317.000000,35159.000000,14317.000000,18791.000000,4550.000000,4550.000000
mean,89630.928267,60007.057381,0.432751,29.116338,23.067692,1990.703200
std,94668.497000,84090.727992,0.250377,15.483885,13.751438,1683.125761
min,1000.000000,100.940000,0.050100,3.500000,6.000000,36.290000
25%,29000.000000,15596.240000,0.236100,14.655000,12.000000,738.237500
50%,64000.000000,36290.090000,0.370000,31.740000,18.000000,1477.315000
75%,111000.000000,66193.265000,0.576200,41.060000,24.000000,2784.105000
max,494000.000000,499905.570000,1.000000,66.970000,60.000000,11379.420000


In [28]:
daf.info()

<class 'pandas.DataFrame'>
RangeIndex: 38909 entries, 0 to 38908
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   producto_id              38909 non-null  str    
 1   user_id                  38909 non-null  str    
 2   tipo_producto            38909 non-null  str    
 3   fecha_apertura           38909 non-null  str    
 4   estatus                  38909 non-null  str    
 5   limite_credito           14317 non-null  float64
 6   saldo_actual             35159 non-null  float64
 7   utilizacion_pct          14317 non-null  float64
 8   tasa_interes_anual       18791 non-null  float64
 9   plazo_meses              4550 non-null   float64
 10  monto_mensualidad        4550 non-null   float64
 11  fecha_ultimo_movimiento  38909 non-null  str    
 12  es_dato_sintetico        38909 non-null  bool   
dtypes: bool(1), float64(6), str(6)
memory usage: 3.6 MB


In [29]:
daf.isnull().sum()

producto_id                    0
user_id                        0
tipo_producto                  0
fecha_apertura                 0
estatus                        0
limite_credito             24592
saldo_actual                3750
utilizacion_pct            24592
tasa_interes_anual         20118
plazo_meses                34359
monto_mensualidad          34359
fecha_ultimo_movimiento        0
es_dato_sintetico              0
dtype: int64

In [30]:
daf.isnull().sum().sort_values(ascending=False)

monto_mensualidad          34359
plazo_meses                34359
limite_credito             24592
utilizacion_pct            24592
tasa_interes_anual         20118
saldo_actual                3750
estatus                        0
tipo_producto                  0
user_id                        0
producto_id                    0
fecha_apertura                 0
fecha_ultimo_movimiento        0
es_dato_sintetico              0
dtype: int64

In [31]:
# 1. Agrupar por tipo de producto y ver cuántos nulos hay en cada uno
analisis_nulos = daf.groupby('tipo_producto')['tasa_interes_anual'].apply(lambda x: x.isnull().sum()).reset_index()
analisis_nulos.columns = ['tipo_producto', 'cantidad_de_nulos']

# 2. Ver también el total de registros por producto para calcular el porcentaje
analisis_nulos['total_registros'] = daf.groupby('tipo_producto')['tasa_interes_anual'].size().values
analisis_nulos['porcentaje_nulos'] = (analisis_nulos['cantidad_de_nulos'] / analisis_nulos['total_registros']) * 100

# 3. Mostrar los productos que tienen nulos ordenados de mayor a menor
print("Análisis de nulos en Tasa de Interés por Producto:")
print(analisis_nulos[analisis_nulos['cantidad_de_nulos'] > 0].sort_values(by='cantidad_de_nulos', ascending=False))

Análisis de nulos en Tasa de Interés por Producto:
     tipo_producto  cantidad_de_nulos  total_registros  porcentaje_nulos
3    cuenta_debito              15025            15025             100.0
7      seguro_vida               2480             2480             100.0
4  cuenta_negocios               1343             1343             100.0
6   seguro_compras               1270             1270             100.0


In [32]:
# Columnas que aceptan el 0 como valor lógico
cols_logicas_cero = ['limite_credito', 'utilizacion_pct', 'monto_mensualidad', 'plazo_meses']
daf[cols_logicas_cero] = daf[cols_logicas_cero].fillna(0)

In [33]:
# Borrar filas donde el saldo es un misterio
daf = daf.dropna(subset=['saldo_actual'])

In [34]:
daf.isnull().sum().sort_values(ascending=False)

tasa_interes_anual         16368
user_id                        0
tipo_producto                  0
fecha_apertura                 0
producto_id                    0
estatus                        0
limite_credito                 0
saldo_actual                   0
utilizacion_pct                0
plazo_meses                    0
monto_mensualidad              0
fecha_ultimo_movimiento        0
es_dato_sintetico              0
dtype: int64

In [35]:
# Rellenar con 0 los nulos de la tasa de interés
daf['tasa_interes_anual'] = daf['tasa_interes_anual'].fillna(0)

# Verificación de seguridad
nulos_restantes = daf['tasa_interes_anual'].isnull().sum()
print(f"Nulos restantes en tasa_interes_anual: {nulos_restantes}")

Nulos restantes en tasa_interes_anual: 0


In [36]:
productos = daf.copy()

In [37]:
import pandas as pd
import numpy as np

productos_limpios = productos.copy()

# Convertir fechas
productos_limpios["fecha_apertura"] = pd.to_datetime(
    productos_limpios["fecha_apertura"], errors="coerce"
)

productos_limpios["fecha_ultimo_movimiento"] = pd.to_datetime(
    productos_limpios["fecha_ultimo_movimiento"], errors="coerce"
)

# Quitar espacios en columnas de texto
columnas_texto = productos_limpios.select_dtypes(include=["object"]).columns

for col in columnas_texto:
    productos_limpios[col] = productos_limpios[col].astype(str).str.strip()

# Reemplazar posibles "nan" convertidos a texto
productos_limpios = productos_limpios.replace("nan", np.nan)

# Crear indicadores antes de rellenar nulos
productos_limpios["tiene_limite_credito"] = productos_limpios["limite_credito"].notna().astype(int)
productos_limpios["tiene_utilizacion"] = productos_limpios["utilizacion_pct"].notna().astype(int)
productos_limpios["tiene_tasa_interes"] = productos_limpios["tasa_interes_anual"].notna().astype(int)
productos_limpios["tiene_plazo"] = productos_limpios["plazo_meses"].notna().astype(int)
productos_limpios["tiene_mensualidad"] = productos_limpios["monto_mensualidad"].notna().astype(int)
productos_limpios["tiene_saldo"] = productos_limpios["saldo_actual"].notna().astype(int)

# Columnas numéricas donde el nulo significa "no aplica"
columnas_no_aplica = [
    "limite_credito",
    "saldo_actual",
    "utilizacion_pct",
    "tasa_interes_anual",
    "plazo_meses",
    "monto_mensualidad"
]

productos_limpios[columnas_no_aplica] = productos_limpios[columnas_no_aplica].fillna(0)

# Rellenar textos faltantes, por seguridad
columnas_texto = productos_limpios.select_dtypes(include=["object"]).columns
productos_limpios[columnas_texto] = productos_limpios[columnas_texto].fillna("sin_dato")

# Si quedara alguna fecha nula, poner fecha controlada
productos_limpios["fecha_apertura"] = productos_limpios["fecha_apertura"].fillna(pd.Timestamp("1900-01-01"))
productos_limpios["fecha_ultimo_movimiento"] = productos_limpios["fecha_ultimo_movimiento"].fillna(pd.Timestamp("1900-01-01"))

# Validación
print("Nulos totales:", productos_limpios.isnull().sum().sum())
productos_limpios.isnull().sum().sort_values(ascending=False).head(20)

Nulos totales: 0


C:\Users\CommodorePlus\AppData\Local\Temp\ipykernel_17376\4264093228.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = productos_limpios.select_dtypes(include=["object"]).columns
C:\Users\CommodorePlus\AppData\Local\Temp\ipykernel_17376\4264093228.py:45: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See ht

producto_id                0
user_id                    0
tipo_producto              0
fecha_apertura             0
estatus                    0
limite_credito             0
saldo_actual               0
utilizacion_pct            0
tasa_interes_anual         0
plazo_meses                0
monto_mensualidad          0
fecha_ultimo_movimiento    0
es_dato_sintetico          0
tiene_limite_credito       0
tiene_utilizacion          0
tiene_tasa_interes         0
tiene_plazo                0
tiene_mensualidad          0
tiene_saldo                0
dtype: int64

In [38]:
ruta_salida = "../data/clean/hey_productos_limpio.csv"

productos_limpios.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"✅ Proceso completado. Archivo guardado en: {ruta_salida}")

✅ Proceso completado. Archivo guardado en: ../data/clean/hey_productos_limpio.csv
